In [ ]:
import numpy as np
import pennylane as qml
import numpy.polynomial.chebyshev as C
import matplotlib.pyplot as plt
import torch
from functools import reduce
import itertools

# ==========================
# Reproducibility
# ==========================
np.random.seed(5)

# ==========================
# Hyperparameters
# ==========================
m = 2
n_qubits = 2
L = 2
Nx_u = 12
Nx_fit = 12
deg_max = 5
blocks = 2
n_samples = 10

# safety: avoid aliasing when accessing +/- frequencies up to deg_max
assert Nx_u > 2 * deg_max + 1, "Need Nx_u > 2*deg_max+1 to avoid FFT aliasing for +/- frequencies."

dev = qml.device("default.qubit", wires=n_qubits)

# ==========================
# Observable
# ==========================
def Observable_local(n, device='cpu', dtype=torch.float32):
    P0 = torch.tensor([[1., 0.],
                       [0., 0.]], device=device, dtype=dtype)
    I2 = torch.eye(2, device=device, dtype=dtype)

    OL = torch.zeros((2**n, 2**n), device=device, dtype=dtype)
    for j in range(n):
        factors = [P0 if k == j else I2 for k in range(n)]
        op_j = reduce(lambda A, B: torch.kron(A, B), factors)
        OL += op_j
    return OL / n

O_L = Observable_local(n_qubits).numpy()

# ==========================
# Encoding
# ==========================
def novel_exp_encoding_multi(x_vec, layer):
    for j in range(m):
        phi = x_vec[j]
        coef = 3 ** layer
        qml.RZ(coef * phi, wires=j)

# ==========================
# Circuits
# ==========================
@qml.qnode(dev)
def circuit_QFM(u_vec, weights):
    qml.templates.StronglyEntanglingLayers(weights[0], wires=range(n_qubits))
    for l in range(L):
        novel_exp_encoding_multi(u_vec, l)
        qml.templates.StronglyEntanglingLayers(weights[l+1], wires=range(n_qubits))
    return qml.expval(qml.Hermitian(O_L, wires=range(n_qubits)))

@qml.qnode(dev)
def circuit_QCM(x_vec, weights, sign_vec):
    qml.templates.StronglyEntanglingLayers(weights[0], wires=range(n_qubits))
    for l in range(L):
        signed = np.arccos(x_vec) * sign_vec
        novel_exp_encoding_multi(signed, l)
        qml.templates.StronglyEntanglingLayers(weights[l+1], wires=range(n_qubits))
    return qml.expval(qml.Hermitian(O_L, wires=range(n_qubits)))

# ==========================
# Helpers: FFT coefficient access with negative frequencies
# ==========================
def fft_idx(w, N):
    """Map integer frequency w (can be negative) to numpy FFT index."""
    return w % N

def c_from_fft(c_fft, w1, w2):
    """Get c_{(w1,w2)} from full 2D FFT array."""
    return c_fft[fft_idx(w1, c_fft.shape[0]), fft_idx(w2, c_fft.shape[1])]

# ==========================
# QFM: compute full 2D FFT coefficients
# ==========================
def qfm_fft_full(weights):
    u = np.linspace(0, 2*np.pi, Nx_u, endpoint=False)
    f = np.zeros((Nx_u, Nx_u), dtype=float)
    for i in range(Nx_u):
        for j in range(Nx_u):
            f[i, j] = circuit_QFM(np.array([u[i], u[j]]), weights)
    c_fft = np.fft.fftn(f) / (Nx_u**2)  # full complex array shape (Nx_u,Nx_u)
    return c_fft

# ==========================
# QCM: 2D Chebyshev fit using chebvander2d + lstsq
# ==========================
def qcm_coeffs_2d_chebfit(weights):
    x = np.linspace(-1, 1, Nx_fit)
    X1, X2 = np.meshgrid(x, x, indexing='ij')
    y = np.zeros_like(X1, dtype=float)

    sign_list = list(itertools.product([-1, 1], repeat=m))

    for i in range(Nx_fit):
        for j in range(Nx_fit):
            xv = np.array([X1[i, j], X2[i, j]])
            y_val = 0.0
            for s in sign_list:
                s_vec = np.array(s)
                y_val += circuit_QCM(xv, weights, s_vec)
            y[i, j] = y_val / (2**m)

    x1_flat = X1.ravel()
    x2_flat = X2.ravel()
    y_flat  = y.ravel()

    V = C.chebvander2d(x1_flat, x2_flat, [deg_max, deg_max])
    coeffs_flat, *_ = np.linalg.lstsq(V, y_flat, rcond=None)
    coeffs = coeffs_flat.reshape(deg_max+1, deg_max+1)

    # Reconstruction validation (fit correctness)
    y_recon = C.chebval2d(X1, X2, coeffs)
    recon_mse = np.mean((y_recon - y)**2)

    return coeffs, recon_mse

# ==========================
# Verify: a_nu from FFT (two ways) and compare with Cheb fit
# ==========================
def a_from_fft_fullsign(c_fft, nu):
    """
    a_nu = sum_{s in {±1}^r} c_{s ⊙ nu}  (zeros stay zero)
    nu: tuple/list length m with nonnegative ints
    """
    nu = np.asarray(nu, dtype=int)
    I = np.where(nu > 0)[0]
    r = len(I)
    if r == 0:
        return c_from_fft(c_fft, 0, 0)

    total = 0.0 + 0.0j
    for s in itertools.product([-1, 1], repeat=r):
        w = np.zeros(m, dtype=int)
        for t, j in enumerate(I):
            w[j] = s[t] * nu[j]
        total += c_from_fft(c_fft, w[0], w[1])
    return total

def a_from_fft_abs(c_fft, nu):
    """
    Compute
        a_nu = sum_{omega : |omega| = nu} c_omega
    using full FFT array.
    """
    nu = np.asarray(nu, dtype=int)

    total = 0.0 + 0.0j

    # enumerate all possible omega in range [-deg_max,deg_max]
    # but only those with |omega|=nu contribute
    for w1 in range(-nu[0], nu[0]+1):
        if abs(w1) != nu[0]:
            continue
        for w2 in range(-nu[1], nu[1]+1):
            if abs(w2) != nu[1]:
                continue

            total += c_from_fft(c_fft, w1, w2)

    return total

def a_from_fft_half_re(c_fft, nu):
    """
    a_nu = 2 * sum_{sigma in {±1}^{r-1}} Re( c_{omega(sigma)} )
    where omega fixes j0=last nonzero component positive.
    """
    nu = np.asarray(nu, dtype=int)
    I = np.where(nu > 0)[0]
    r = len(I)
    if r == 0:
        return c_from_fft(c_fft, 0, 0).real  # should be real

    j0 = I[-1]  # last nonzero
    J = [j for j in I if j != j0]  # remaining r-1 indices

    total_re = 0.0
    for sigma in itertools.product([-1, 1], repeat=r-1):
        w = np.zeros(m, dtype=int)
        w[j0] = nu[j0]
        for t, j in enumerate(J):
            w[j] = sigma[t] * nu[j]
        total_re += np.real(c_from_fft(c_fft, w[0], w[1]))
    return 2.0 * total_re

# ==========================
# Monte Carlo loop
# ==========================
coeffs_QFM = []
coeffs_QCM = []
var_checks = []  # store per-sample verification errors

for sample in range(n_samples):

    weights = np.random.uniform(
        0, 2*np.pi,
        size=(L+1, blocks, n_qubits, 3)
    )

    # --- QFM full FFT coefficients ---
    c_fft = qfm_fft_full(weights)

    # --- QCM Chebyshev fit coefficients ---
    a_cheb, recon_mse = qcm_coeffs_2d_chebfit(weights)

    # --- build a_nu from FFT two ways on grid 0..deg_max ---
    a_full = np.zeros((deg_max+1, deg_max+1), dtype=complex)
    a_abs = np.zeros((deg_max+1, deg_max+1), dtype=complex)
    a_half = np.zeros((deg_max+1, deg_max+1), dtype=float)

    for n1 in range(deg_max+1):
        for n2 in range(deg_max+1):
            nu = (n1, n2)
            a_abs[n1,n2]  = a_from_fft_abs(c_fft, nu)
            a_full[n1, n2] = a_from_fft_fullsign(c_fft, nu)
            a_half[n1, n2] = a_from_fft_half_re(c_fft, nu)

    # --- verification metrics ---
    # 1) verify your identity: a_full == a_half (up to numerical error)
    # err_id = np.max(np.abs(a_full - a_half))
    err_id = np.max(np.abs(a_abs - a_half))

    # 2) compare to cheb fit coefficients (should match approximately; errors depend on Nx_fit/deg_max conditioning)
    err_full_cheb = np.max(np.abs(a_abs.real - a_cheb))
    err_half_cheb = np.max(np.abs(a_half - a_cheb))

    print(f"[sample {sample:02d}] recon MSE (Cheb fit): {recon_mse:.3e} | "
          f"max|a_full - a_half|: {err_id:.3e} | "
          f"max|a_full - a_cheb|: {err_full_cheb:.3e} | "
          f"max|a_half - a_cheb|: {err_half_cheb:.3e}")

    # store for variance plot (keep the "a" objects you care about)
    coeffs_QFM.append(a_abs)      # these are the collapsed a_nu computed from FFT full-sign sum
    coeffs_QCM.append(a_cheb)      # these are the fitted cheb coefficients

    var_checks.append((recon_mse, err_id, err_full_cheb, err_half_cheb))

coeffs_QFM = np.array(coeffs_QFM)  # (n_samples, deg+1, deg+1), complex (should be ~real)
coeffs_QCM = np.array(coeffs_QCM)  # (n_samples, deg+1, deg+1), real

# ==========================
# Variance (keep your original plotting intent)
# ==========================
var_QFM = np.var(np.real(coeffs_QFM), axis=0)
var_QCM = np.var(coeffs_QCM, axis=0)

# ==========================
# Plot
# ==========================
import matplotlib as mpl
mpl.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 1.2,
})
mpl.rcParams["mathtext.fontset"] = "dejavusans"

plt.figure(figsize=(6,5))
plt.imshow(var_QFM, origin="lower", aspect="auto")
plt.colorbar(label="Variance")
plt.xlabel(r"$\nu_1$")
plt.ylabel(r"$\nu_2$")
plt.title("Variance of a_nu from FFT collapse (QFM -> Chebyshev)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,5))
plt.imshow(var_QCM, origin="lower", aspect="auto")
plt.colorbar(label="Variance")
plt.xlabel(r"$\nu_1$")
plt.ylabel(r"$\nu_2$")
plt.title("Variance of a_nu from Chebyshev fit (QCM)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,5))
plt.imshow(var_QFM - var_QCM, origin="lower", aspect="auto")
plt.colorbar(label="Var(diff)")
plt.xlabel(r"$\nu_1$")
plt.ylabel(r"$\nu_2$")
plt.title("Variance difference")
plt.tight_layout()
plt.show()